<a href="https://colab.research.google.com/github/cwilson-su/Large-Scale-Data-Analytics-Labs/blob/main/LAB2_2026_Fenetres_avec_duckdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

révision : 02/10/2026

# TP2 — Fonctions de fenêtrage (`OVER`) avec DuckDB — ÉNONCÉ

* **Données** : un petit exemple (visites de photos), puis le jeu de données Foursquare (New York)
* **Moteur SQL** : DuckDB

Ce TP met en pratique la partie *Fenêtrage* du cours 2 (*Analyse multidimensionnelle en SQL*) :
`row_number`, `rank`, `dense_rank`, `PARTITION BY`, fenêtres croissantes et glissantes (`ROWS`, `RANGE`).

# Préparation

Lire la [documentation DuckDB SQL](https://duckdb.org/docs/sql/introduction.html) :
* [Functions](https://duckdb.org/docs/sql/functions/overview)
* [Window functions](https://duckdb.org/docs/sql/window_functions)

Affichage convivial des résultats d'une requête

In [1]:
try:
    from itables import init_notebook_mode
except ImportError:
    !pip -q install itables
    from itables import init_notebook_mode

init_notebook_mode(all_interactive=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 23.9 MB/s eta 0:00:00


In [2]:
import pandas as pd

def display(query, n=30):
    """Affiche les n premiers tuples d'une requête DuckDB sous forme de tableau."""
    return query.limit(n).df()

# Variante pour Colab uniquement :
# from google.colab import data_table
# def display(query, n=100):
#     return data_table.DataTable(query.limit(n).df(), include_index=False, num_rows_per_page=10)

## Installer et importer DuckDB

In [3]:
try:
    import duckdb
except ImportError:
    !pip install duckdb
    import duckdb

Définir la commande magique **`%%sql`** pour écrire plus simplement une requête SQL dans une cellule.

In [4]:
from IPython.core.magic import register_line_cell_magic

def removeComments(query):
    """Supprime les lignes de commentaire SQL (commençant par --)."""
    result = ""
    for line in query.split('\n'):
        if not line.strip().startswith("--"):
            result += line + "\n"
    return result

@register_line_cell_magic
def sql(line, cell=None):
    "Exécute une ou plusieurs requêtes SQL séparées par ';'. Usage : %%sql"
    val = cell if cell is not None else line
    derniere = None
    est_requete = False
    for r in removeComments(val).split(";"):
        r = r.strip()
        if len(r) > 2:
            derniere = db.sql(r)
            est_requete = r.lower().startswith(('select', 'with', 'from'))
    if est_requete:
        return display(derniere)
    print('ok')

print("fonction définie")

fonction définie


## Démarrer DuckDB

In [5]:
db = duckdb.connect(':memory:')
# Les dates Foursquare sont en UTC : on fixe le fuseau horaire de la session
db.sql("SET TimeZone = 'UTC'")

# vérifier que le service fonctionne
display(db.sql("SELECT 'hello' AS col"))

Loading ITables v2.9.1 from the internet... (need help?)
🔒ⓘcol
hello


In [6]:
def display_schema(table):
    return display(db.sql(f"""
        SELECT column_name, data_type
        FROM duckdb_columns
        WHERE table_name = '{table}'
    """))

def display_all_views():
    return display(db.sql("SELECT view_name, sql, temporary FROM duckdb_views"))

print("fonctions définies")

fonctions définies


# Partie 1 — Premier exemple

Un premier exemple pour comprendre les fonctions de fenêtrage. Ce sont les données utilisées dans le cours.

## Les données

### Les visites

In [7]:
visite_attrs = ["photoId", "personId", "date", "lon", "lat", "note"]
visite_tuples = [("p1",  "Bob",    "03/09/2020", 41.5, 12.8, 3),
                 ("p2",  "Alice",  "01/09/2020", 41.6, 12.8, 5),
                 ("p3",  "Bob",    "04/09/2020", 41.6, 13.2, 1),
                 ("p4",  "Bob",    "04/09/2020", 40.1, 12.0, 2),
                 ("p5",  "Alice",  "04/09/2020", 40.1, 12.0, 2),
                 ("p6",  "Alice",  "05/09/2020", 41.6, 12.7, 4),
                 ("p7",  "Alice",  "05/10/2020", 41.8, 12.8, 4),
                 ("p8",  "Carole", "25/12/2019", 30.1, 10.1, 2),
                 ("p9",  "David",  "25/12/2019", 30.1, 10.1, 1),
                 ("p10", "Eva",    "25/12/2019", 30.1, 10.1, 5),
                 ("p11", "Eva",    "26/12/2019", 31.1, 10.1, 3),
                 ("p12", "Alice",  "01/02/2020", 32.1, 12.1, 3),
                 ("p13", "Bob",    "01/02/2020", 32.1, 12.1, 5),
                 ("p14", "Carole", "01/02/2020", 32.1, 12.1, 2),
                 ("p15", "Carole", "11/11/2019", 49.1, 10.1, 1),
                 ("p16", "Alice",  "25/12/2019", 30.1, 10.1, 4),
                 ("p17", "Alice",  "02/02/2020", 49.1, 10.1, 5),
                ]

visites_df = pd.DataFrame(visite_tuples, columns=visite_attrs)

db.sql("""
DROP TABLE IF EXISTS Visites CASCADE;

CREATE TABLE Visites AS
SELECT photoId,
       personId,
       strptime(date, '%d/%m/%Y') AS date_num,
       extract(year FROM date_num) AS année,
       extract(month FROM date_num) AS mois,
       lon,
       lat,
       note
FROM visites_df""")

Visites = db.table("Visites")
display(Visites)

photoId personId   date_num  année  mois   lon   lat  note
0       p1      Bob 2020-09-03   2020     9  41.5  12.8     3
1       p2    Alice 2020-09-01   2020     9  41.6  12.8     5
2       p3      Bob 2020-09-04   2020     9  41.6  13.2     1
3       p4      Bob 2020-09-04   2020     9  40.1  12.0     2
4       p5    Alice 2020-09-04   2020     9  40.1  12.0     2
5       p6    Alice 2020-09-05   2020     9  41.6  12.7     4
6       p7    Alice 2020-10-05   2020    10  41.8  12.8     4
7       p8   Carole 2019-12-25   2019    12  30.1  10.1     2
8       p9    David 2019-12-25   2019    12  30.1  10.1     1
9      p10      Eva 2019-12-25   2019    12  30.1  10.1     5
10     p11      Eva 2019-12-26   2019    12  31.1  10.1     3
11     p12    Alice 2020-02-01   2020     2  32.1  12.1     3
12     p13      Bob 2020-02-01   2020     2  32.1  12.1     5
13     p14   Carole 2020-02-01   2020     2  32.1  12.1     2
14     p15   Carole 2019-11-11   2019    11  49.1  10.1     1
15     p16    Alice 2019-12-25   2019    12  30.1  10.1     4
16     p17    Alice 2020-02-02   2020     2  49.1  10.1     5

In [8]:
display_schema("Visites")

column_name  data_type
0     photoId    VARCHAR
1    personId    VARCHAR
2    date_num  TIMESTAMP
3       année     BIGINT
4        mois     BIGINT
5         lon     DOUBLE
6         lat     DOUBLE
7        note     BIGINT

Exemple de cellule utilisant la commande `%%sql` :

In [9]:
%%sql
SELECT *
FROM Visites
LIMIT 3

photoId personId   date_num  année  mois   lon   lat  note
0      p1      Bob 2020-09-03   2020     9  41.5  12.8     3
1      p2    Alice 2020-09-01   2020     9  41.6  12.8     5
2      p3      Bob 2020-09-04   2020     9  41.6  13.2     1

### Les lieux

In [10]:
lieux_attributs = ["lon", "lat", "ville", "pays"]

lieux_tuples = [(41.5, 12.8, "Aix", "France"),
                (41.6, 12.8, "Aix", "France"),
                (41.6, 12.7, "Nice", "France"),
                (41.8, 12.8, "Marseille", "France"),
                (41.6, 13.2, "Rome", "Italie"),
                (40.1, 12.0, "Oslo", "Norvege"),
                (30.1, 10.1, "Paris", "France"),
                (31.1, 10.1, "StDenis", "France"),
                (32.1, 12.1, "Lille", "France"),
                (49.1, 10.1, "Pau", "France")]

lieux_df = pd.DataFrame(lieux_tuples, columns=lieux_attributs)
db.sql("""
DROP TABLE IF EXISTS Lieux;
CREATE TABLE Lieux AS
SELECT ville, pays, lon, lat FROM lieux_df""")

Lieux = db.table("Lieux")
display(Lieux)

ville     pays   lon   lat
0        Aix   France  41.5  12.8
1        Aix   France  41.6  12.8
2       Nice   France  41.6  12.7
3  Marseille   France  41.8  12.8
4       Rome   Italie  41.6  13.2
5       Oslo  Norvege  40.1  12.0
6      Paris   France  30.1  10.1
7    StDenis   France  31.1  10.1
8      Lille   France  32.1  12.1
9        Pau   France  49.1  10.1

### Les professions

In [11]:
profession_attributs = ["personID", "profession"]

profession_tuples = [("Alice", "Architecte"),
                     ("Bob", "Data science"),
                     ("Carole", "Data science"),
                     ("David", "Architecte"),
                     ("Eva", "Vendeuse"),
                     ("Franck", "Vendeur"),
                     ("Greta", "Economiste")]

profession_df = pd.DataFrame(profession_tuples, columns=profession_attributs)
db.sql("""
DROP TABLE IF EXISTS Professions;
CREATE TABLE Professions AS
SELECT * FROM profession_df""")

Professions = db.table("Professions")
display(Professions)

personID    profession
0    Alice    Architecte
1      Bob  Data science
2   Carole  Data science
3    David    Architecte
4      Eva      Vendeuse
5   Franck       Vendeur
6    Greta    Economiste

### Le détail des visites

Définir une vue contenant le détail des visites.

In [12]:
db.sql("""
CREATE OR REPLACE VIEW VisitesDetail AS
SELECT v.photoId, p.personID, p.profession, v.date_num, v.mois, v.année,
       v.lon, v.lat, l.ville, l.pays, v.note
FROM Visites v
JOIN Lieux l ON (v.lon = l.lon AND v.lat = l.lat)
JOIN Professions p ON (v.personId = p.personId)
""")

# Même requête écrite avec un NATURAL JOIN :
# SELECT ... FROM Visites v NATURAL JOIN Lieux l NATURAL JOIN Professions p

VisitesDetail = db.view('VisitesDetail')
display(VisitesDetail)

photoId personID    profession   date_num  mois  année   lon   lat  \
0       p1      Bob  Data science 2020-09-03     9   2020  41.5  12.8   
1       p2    Alice    Architecte 2020-09-01     9   2020  41.6  12.8   
2       p3      Bob  Data science 2020-09-04     9   2020  41.6  13.2   
3       p4      Bob  Data science 2020-09-04     9   2020  40.1  12.0   
4       p5    Alice    Architecte 2020-09-04     9   2020  40.1  12.0   
5       p6    Alice    Architecte 2020-09-05     9   2020  41.6  12.7   
6       p7    Alice    Architecte 2020-10-05    10   2020  41.8  12.8   
7       p8   Carole  Data science 2019-12-25    12   2019  30.1  10.1   
8       p9    David    Architecte 2019-12-25    12   2019  30.1  10.1   
9      p10      Eva      Vendeuse 2019-12-25    12   2019  30.1  10.1   
10     p11      Eva      Vendeuse 2019-12-26    12   2019  31.1  10.1   
11     p12    Alice    Architecte 2020-02-01     2   2020  32.1  12.1   
12     p13      Bob  Data science 2020-02-01     2   2020  32.1  12.1   
13     p14   Carole  Data science 2020-02-01     2   2020  32.1  12.1   
14     p15   Carole  Data science 2019-11-11    11   2019  49.1  10.1   
15     p16    Alice    Architecte 2019-12-25    12   2019  30.1  10.1   
16     p17    Alice    Architecte 2020-02-02     2   2020  49.1  10.1   

        ville     pays  note  
0         Aix   France     3  
1         Aix   France     5  
2        Rome   Italie     1  
3        Oslo  Norvege     2  
4        Oslo  Norvege     2  
5        Nice   France     4  
6   Marseille   France     4  
7       Paris   France     2  
8       Paris   France     1  
9       Paris   France     5  
10    StDenis   France     3  
11      Lille   France     3  
12      Lille   France     5  
13      Lille   France     2  
14        Pau   France     1  
15      Paris   France     4  
16        Pau   France     5

## Exemples de requêtes avec fenêtre

Ces requêtes sont celles du cours : exécutez-les et observez le résultat.

### Numéroter les tuples : `row_number()`

In [13]:
%%sql
SELECT v.personId, v.photoId, v.note,
       row_number() OVER (ORDER BY personId, photoId) AS numeroPhoto
FROM Visites v

personId photoId  note  numeroPhoto
0     Alice     p12     3            1
1     Alice     p16     4            2
2     Alice     p17     5            3
3     Alice      p2     5            4
4     Alice      p5     2            5
5     Alice      p6     4            6
6     Alice      p7     4            7
7       Bob      p1     3            8
8       Bob     p13     5            9
9       Bob      p3     1           10
10      Bob      p4     2           11
11   Carole     p14     2           12
12   Carole     p15     1           13
13   Carole      p8     2           14
14    David      p9     1           15
15      Eva     p10     5           16
16      Eva     p11     3           17

Remarque : `photoId` est une chaîne de caractères, l'ordre est donc lexicographique (`p12 < p16 < p2`).

### Classer les visites par note décroissante : `rank()`

In [14]:
%%sql
SELECT v.personId, v.photoId, v.note,
       rank() OVER (ORDER BY note DESC) AS rang
FROM Visites v

personId photoId  note  rang
0     Alice      p2     5     1
1       Eva     p10     5     1
2       Bob     p13     5     1
3     Alice     p17     5     1
4     Alice      p6     4     5
5     Alice      p7     4     5
6     Alice     p16     4     5
7       Bob      p1     3     8
8       Eva     p11     3     8
9     Alice     p12     3     8
10      Bob      p4     2    11
11    Alice      p5     2    11
12   Carole      p8     2    11
13   Carole     p14     2    11
14      Bob      p3     1    15
15    David      p9     1    15
16   Carole     p15     1    15

### Requête top-K
Le top 5 des visites ayant les meilleures notes.
Il peut y avoir plus de 5 visites dans le résultat à cause des **ex æquo**.

In [15]:
%%sql
WITH Classement AS (
  SELECT v.personId, v.photoId, v.note,
         rank() OVER (ORDER BY note DESC) AS rang
  FROM Visites v
)
SELECT *
FROM Classement
WHERE rang <= 5
ORDER BY rang

personId photoId  note  rang
0    Alice      p2     5     1
1      Bob     p13     5     1
2    Alice     p17     5     1
3      Eva     p10     5     1
4    Alice      p6     4     5
5    Alice      p7     4     5
6    Alice     p16     4     5

### Classement dense : `dense_rank()`

In [16]:
%%sql
SELECT v.personId, v.photoId, v.note,
       dense_rank() OVER (ORDER BY note DESC) AS rang_dense
FROM Visites v

personId photoId  note  rang_dense
0     Alice      p2     5           1
1       Eva     p10     5           1
2       Bob     p13     5           1
3     Alice     p17     5           1
4     Alice      p6     4           2
5     Alice      p7     4           2
6     Alice     p16     4           2
7       Bob      p1     3           3
8       Eva     p11     3           3
9     Alice     p12     3           3
10      Bob      p4     2           4
11    Alice      p5     2           4
12   Carole      p8     2           4
13   Carole     p14     2           4
14      Bob      p3     1           5
15    David      p9     1           5
16   Carole     p15     1           5

### Fenêtrage et partitionnement : `PARTITION BY`

Une partition regroupe les tuples ayant la même valeur pour les attributs de partitionnement.
On obtient une fenêtre par partition.

Le classement **par ville** des visites, de la mieux notée à la moins bien notée

In [17]:
%%sql
SELECT v.photoId, v.personID, v.ville, v.note,
       rank() OVER (PARTITION BY ville ORDER BY note DESC) AS rang
FROM VisitesDetail v
ORDER BY ville, rang

photoId personID      ville  note  rang
0       p2    Alice        Aix     5     1
1       p1      Bob        Aix     3     2
2      p13      Bob      Lille     5     1
3      p12    Alice      Lille     3     2
4      p14   Carole      Lille     2     3
5       p7    Alice  Marseille     4     1
6       p6    Alice       Nice     4     1
7       p4      Bob       Oslo     2     1
8       p5    Alice       Oslo     2     1
9      p10      Eva      Paris     5     1
10     p16    Alice      Paris     4     2
11      p8   Carole      Paris     2     3
12      p9    David      Paris     1     4
13     p17    Alice        Pau     5     1
14     p15   Carole        Pau     1     2
15      p3      Bob       Rome     1     1
16     p11      Eva    StDenis     3     1

Le classement **par pays** des villes ayant le plus de visites

In [18]:
%%sql
SELECT pays, ville, count(*) AS nbVisite,
       rank() OVER (PARTITION BY pays ORDER BY count(*) DESC) AS rang
FROM VisitesDetail v
GROUP BY pays, ville
ORDER BY pays, rang

pays      ville  nbVisite  rang
0   France      Paris         4     1
1   France      Lille         3     2
2   France        Aix         2     3
3   France        Pau         2     3
4   France  Marseille         1     5
5   France       Nice         1     5
6   France    StDenis         1     5
7   Italie       Rome         1     1
8  Norvege       Oslo         2     1

La fenêtre est évaluée **après** le `GROUP BY` : elle peut donc utiliser `count(*)`.

## Questions

### Question 1 : temps écoulé depuis la première visite

Pour chaque visite de chaque personne, calculer le temps écoulé depuis la **première** visite de cette personne.
Le temps écoulé est un intervalle de temps, obtenu par différence entre deux dates ; il s'exprime en jours.

Schéma attendu : `(photoId, personId, date_num, datePremiereVisite, jours_ecoules)`, trié par personne puis par date.

*Astuce* : commencer par compléter chaque visite avec la date de la première visite de la personne.

In [20]:
%%sql
WITH VisitesAvecPremiere AS (
    SELECT
        photoId,
        personId,
        date_num,
        MIN(date_num) OVER (PARTITION BY personId) AS datePremiereVisite
    FROM Visites
)
SELECT
    photoId,
    personId,
    date_num,
    datePremiereVisite,
    datediff('day', datePremiereVisite, date_num) AS jours_ecoules
FROM VisitesAvecPremiere
ORDER BY personId, date_num;

photoId personId   date_num datePremiereVisite  jours_ecoules
0      p16    Alice 2019-12-25         2019-12-25              0
1      p12    Alice 2020-02-01         2019-12-25             38
2      p17    Alice 2020-02-02         2019-12-25             39
3       p2    Alice 2020-09-01         2019-12-25            251
4       p5    Alice 2020-09-04         2019-12-25            254
5       p6    Alice 2020-09-05         2019-12-25            255
6       p7    Alice 2020-10-05         2019-12-25            285
7      p13      Bob 2020-02-01         2020-02-01              0
8       p1      Bob 2020-09-03         2020-02-01            215
9       p4      Bob 2020-09-04         2020-02-01            216
10      p3      Bob 2020-09-04         2020-02-01            216
11     p15   Carole 2019-11-11         2019-11-11              0
12      p8   Carole 2019-12-25         2019-11-11             44
13     p14   Carole 2020-02-01         2019-11-11             82
14      p9    David 2019-12-25         2019-12-25              0
15     p10      Eva 2019-12-25         2019-12-25              0
16     p11      Eva 2019-12-26         2019-12-25              1

### Question 2 : fenêtre glissante de taille fixe — visite précédente

Pour chaque visite de chaque personne, calculer le nombre de jours écoulés depuis la visite **précédente** de cette personne.
Si une personne a effectué plusieurs visites le même jour, on considère ces visites dans l'ordre de leur `photoId`.
Pour la première visite d'une personne, `precedenteDate` et `nbJours` valent `NULL`.

Schéma attendu : `(photoId, personId, date_num, precedenteDate, nbJours)`

*Astuce* : la fonction `datediff('day', d1, d2)` donne le nombre de jours entre deux dates.

In [21]:
%%sql
WITH VisitesAvecPrecedente AS (
    SELECT
        photoId,
        personId,
        date_num,
        LAG(date_num) OVER (
            PARTITION BY personId
            ORDER BY date_num, photoId
        ) AS precedenteDate
    FROM Visites
)
SELECT
    photoId,
    personId,
    date_num,
    precedenteDate,
    datediff('day', precedenteDate, date_num) AS nbJours
FROM VisitesAvecPrecedente
ORDER BY personId, date_num, photoId;

photoId personId   date_num precedenteDate  nbJours
0      p16    Alice 2019-12-25            NaT     <NA>
1      p12    Alice 2020-02-01     2019-12-25       38
2      p17    Alice 2020-02-02     2020-02-01        1
3       p2    Alice 2020-09-01     2020-02-02      212
4       p5    Alice 2020-09-04     2020-09-01        3
5       p6    Alice 2020-09-05     2020-09-04        1
6       p7    Alice 2020-10-05     2020-09-05       30
7      p13      Bob 2020-02-01            NaT     <NA>
8       p1      Bob 2020-09-03     2020-02-01      215
9       p3      Bob 2020-09-04     2020-09-03        1
10      p4      Bob 2020-09-04     2020-09-04        0
11     p15   Carole 2019-11-11            NaT     <NA>
12      p8   Carole 2019-12-25     2019-11-11       44
13     p14   Carole 2020-02-01     2019-12-25       38
14      p9    David 2019-12-25            NaT     <NA>
15     p10      Eva 2019-12-25            NaT     <NA>
16     p11      Eva 2019-12-26     2019-12-25        1

### Question 3 : semaine glissante

Pour chaque jour où une personne a effectué au moins une visite, calculer le nombre de visites de cette personne
sur les 7 derniers jours (le jour courant et les 6 jours précédents).

Schéma attendu : `(personId, date_num, nbVisites_7jours)`

*Astuce* : une fenêtre `RANGE` est définie par un écart de valeur, par exemple `INTERVAL 6 DAYS`.

In [22]:
%%sql
WITH VisitesParJour AS (
    -- Step 1-count how many visits each person made on each specific date
    SELECT
        personId,
        date_num,
        COUNT(*) AS nbVisitesJour
    FROM Visites
    GROUP BY personId, date_num
)
SELECT
    personId,
    date_num,
    SUM(nbVisitesJour) OVER (
        PARTITION BY personId
        ORDER BY date_num
        RANGE BETWEEN INTERVAL 6 DAY PRECEDING AND CURRENT ROW
    ) AS nbVisites_7jours
FROM VisitesParJour
ORDER BY personId, date_num;

personId   date_num  nbVisites_7jours
0     Alice 2019-12-25               1.0
1     Alice 2020-02-01               1.0
2     Alice 2020-02-02               2.0
3     Alice 2020-09-01               1.0
4     Alice 2020-09-04               2.0
5     Alice 2020-09-05               3.0
6     Alice 2020-10-05               1.0
7       Bob 2020-02-01               1.0
8       Bob 2020-09-03               1.0
9       Bob 2020-09-04               3.0
10   Carole 2019-11-11               1.0
11   Carole 2019-12-25               1.0
12   Carole 2020-02-01               1.0
13    David 2019-12-25               1.0
14      Eva 2019-12-25               1.0
15      Eva 2019-12-26               2.0

### Question 4 : trajets

On veut enrichir les visites avec une information de trajet.
Un trajet regroupe des visites consécutives d'une même personne telles que deux visites consécutives
sont séparées d'**au plus 7 jours**. Une visite séparée de plus de 7 jours de la précédente commence un nouveau trajet.
Les trajets d'une personne sont numérotés 1, 2, 3, … par ordre chronologique.

Schéma attendu : `(photoId, personId, date_num, nbJours, trajet)`

*Astuces* :
* réutiliser la réponse à la question 2 ;
* une expression `if(condition, 1, 0)` (ou `CASE WHEN … THEN 1 ELSE 0 END`) indique si la visite commence un nouveau trajet ;
* une somme cumulée de ces indicateurs donne le numéro de trajet.

In [23]:
%%sql
WITH VisitesAvecDelai AS (
    -- Step 1-Calculate the delay (nbJours) from the previous visit
    SELECT
        photoId,
        personId,
        date_num,
        datediff('day',
            LAG(date_num) OVER (
                PARTITION BY personId
                ORDER BY date_num, photoId
            ),
            date_num
        ) AS nbJours
    FROM Visites
),
VisitesAvecIndicateur AS (
    -- Step 2-mark visits that start a NEW trip with 1, otherwise 0
    SELECT
        photoId,
        personId,
        date_num,
        nbJours,
        CASE
            WHEN nbJours IS NULL OR nbJours > 7 THEN 1
            ELSE 0
        END AS debutTrajet
    FROM VisitesAvecDelai
)
-- Step 3-Compute cumulative sum of trip indicators to form the trip ID
SELECT
    photoId,
    personId,
    date_num,
    nbJours,
    SUM(debutTrajet) OVER (
        PARTITION BY personId
        ORDER BY date_num, photoId
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS trajet
FROM VisitesAvecIndicateur
ORDER BY personId, date_num, photoId;

photoId personId   date_num  nbJours  trajet
0      p16    Alice 2019-12-25     <NA>     1.0
1      p12    Alice 2020-02-01       38     2.0
2      p17    Alice 2020-02-02        1     2.0
3       p2    Alice 2020-09-01      212     3.0
4       p5    Alice 2020-09-04        3     3.0
5       p6    Alice 2020-09-05        1     3.0
6       p7    Alice 2020-10-05       30     4.0
7      p13      Bob 2020-02-01     <NA>     1.0
8       p1      Bob 2020-09-03      215     2.0
9       p3      Bob 2020-09-04        1     2.0
10      p4      Bob 2020-09-04        0     2.0
11     p15   Carole 2019-11-11     <NA>     1.0
12      p8   Carole 2019-12-25       44     2.0
13     p14   Carole 2020-02-01       38     3.0
14      p9    David 2019-12-25     <NA>     1.0
15     p10      Eva 2019-12-25     <NA>     1.0
16     p11      Eva 2019-12-26        1     1.0

### Question 4b (bonus) : trajets d'au plus 3 jours

On ajoute une condition : un trajet ne peut pas durer plus de **3 jours** au total.
Une visite commence donc un nouveau trajet si elle est séparée de plus de 7 jours de la visite précédente,
**ou** si elle a lieu plus de 3 jours après la première visite du trajet en cours.

Schéma attendu : `(photoId, personId, date_num, debutTrajet, trajet)`

*Question préalable* : pourquoi ne peut-on pas résoudre cette question avec une simple somme cumulée comme à la question 4 ?

In [24]:
%%sql
--- AI ANSWER
WITH RECURSIVE
VisitesOrdonnees AS (
    -- Step 1: Number each person's visits sequentially 1, 2, 3...
    SELECT
        photoId,
        personId,
        date_num,
        ROW_NUMBER() OVER (
            PARTITION BY personId
            ORDER BY date_num, photoId
        ) AS rn
    FROM Visites
),
TrajetsRecursifs AS (
    -- Step 2a: Base Case (First visit for each person: rn = 1)
    SELECT
        photoId,
        personId,
        date_num,
        rn,
        date_num AS dateDebutTrajet,
        1 AS debutTrajet,
        1 AS trajet
    FROM VisitesOrdonnees
    WHERE rn = 1

    UNION ALL

    -- Step 2b: Recursive Step (Evaluate visit 'rn' using result from 'rn - 1')
    SELECT
        curr.photoId,
        curr.personId,
        curr.date_num,
        curr.rn,
        -- New trip start date if condition met, else keep previous trip start date
        CASE
            WHEN datediff('day', prev.date_num, curr.date_num) > 7
              OR datediff('day', prev.dateDebutTrajet, curr.date_num) > 3
            THEN curr.date_num
            ELSE prev.dateDebutTrajet
        END AS dateDebutTrajet,

        -- Flag if this visit starts a new trip
        CASE
            WHEN datediff('day', prev.date_num, curr.date_num) > 7
              OR datediff('day', prev.dateDebutTrajet, curr.date_num) > 3
            THEN 1
            ELSE 0
        END AS debutTrajet,

        -- Increment trip ID if new trip, else keep current trip ID
        CASE
            WHEN datediff('day', prev.date_num, curr.date_num) > 7
              OR datediff('day', prev.dateDebutTrajet, curr.date_num) > 3
            THEN prev.trajet + 1
            ELSE prev.trajet
        END AS trajet
    FROM VisitesOrdonnees curr
    JOIN TrajetsRecursifs prev
      ON curr.personId = prev.personId
     AND curr.rn = prev.rn + 1
)
SELECT
    photoId,
    personId,
    date_num,
    debutTrajet,
    trajet
FROM TrajetsRecursifs
ORDER BY personId, date_num, photoId;

photoId personId   date_num  debutTrajet  trajet
0      p16    Alice 2019-12-25            1       1
1      p12    Alice 2020-02-01            1       2
2      p17    Alice 2020-02-02            0       2
3       p2    Alice 2020-09-01            1       3
4       p5    Alice 2020-09-04            0       3
5       p6    Alice 2020-09-05            1       4
6       p7    Alice 2020-10-05            1       5
7      p13      Bob 2020-02-01            1       1
8       p1      Bob 2020-09-03            1       2
9       p3      Bob 2020-09-04            0       2
10      p4      Bob 2020-09-04            0       2
11     p15   Carole 2019-11-11            1       1
12      p8   Carole 2019-12-25            1       2
13     p14   Carole 2020-02-01            1       3
14      p9    David 2019-12-25            1       1
15     p10      Eva 2019-12-25            1       1
16     p11      Eva 2019-12-26            0       1

# Partie 2 — Données Foursquare

Données de *check-in* issues du réseau social Foursquare (ville de New York, avril 2012 – février 2013).
Source : [Kaggle](https://www.kaggle.com/datasets/chetanism/foursquare-nyc-and-tokyo-checkin-dataset).

In [25]:
import os
local_dir = os.environ["HOME"] + "/data"
print("local_dir is", local_dir)
os.makedirs(local_dir, exist_ok=True)
os.listdir(local_dir)

local_dir is /root/data


[]

URL d'accès aux jeux de données

In [26]:
from urllib import request
import zipfile

# URL du dossier PUBLIC_DATASET contenant des fichiers de données pour les TP
PUBLIC_DATASET_URL = "https://gitlab.lip6.fr/naacke/tp/-/raw/main/dataset"

# en cas de problème de téléchargement, aller directement sur l'URL ci-dessous
print("Les datasets disponibles :", PUBLIC_DATASET_URL.replace("/raw/", "/tree/"))


def download_file(url, local_path):
    if not os.path.exists(local_path):
        print("downloading", url, "to", local_path)
        request.urlretrieve(url, local_path)
    else:
        print(local_path, "already exists")


def unzip_file(local_dir, file):
    zip_filepath = os.path.join(local_dir, file)
    try:
        with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
            do_extract = False
            for f in zip_ref.namelist():
                if not os.path.exists(os.path.join(local_dir, f)):
                    do_extract = True
                    break
                print(os.path.join(local_dir, f), "unzipped file already exists")
            if do_extract:
                print("unzip", zip_filepath)
                zip_ref.extractall(local_dir)
    except Exception as e:
        print(f"Une erreur est survenue : {e}")

Les datasets disponibles : https://gitlab.lip6.fr/naacke/tp/-/tree/main/dataset


In [27]:
zip_filename = "dataset_TSMC2014_NYC.zip"
download_file(f"{PUBLIC_DATASET_URL}/foursquare/{zip_filename}", os.path.join(local_dir, zip_filename))
unzip_file(local_dir, zip_filename)

os.listdir(local_dir)

downloading https://gitlab.lip6.fr/naacke/tp/-/raw/main/dataset/foursquare/dataset_TSMC2014_NYC.zip to /root/data/dataset_TSMC2014_NYC.zip
unzip /root/data/dataset_TSMC2014_NYC.zip


['dataset_TSMC2014_NYC.csv', 'dataset_TSMC2014_NYC.zip']

In [28]:
# afficher les premières lignes du fichier
with open(local_dir + "/dataset_TSMC2014_NYC.csv", 'r') as data_file:
    for i in range(4):
        print(data_file.readline().strip())

userId,venueId,venueCategoryId,venueCategory,latitude,longitude,timezoneOffset,utcTimestamp
470,49bbd6c0f964a520f4531fe3,4bf58dd8d48988d127951735,Arts & Crafts Store,40.71981038,-74.00258103,-240,Tue Apr 03 18:00:09 +0000 2012
979,4a43c0aef964a520c6a61fe3,4bf58dd8d48988d1df941735,Bridge,40.60679958,-74.04416981,-240,Tue Apr 03 18:00:25 +0000 2012
69,4c5cc7b485a1e21e00d35711,4bf58dd8d48988d103941735,Home (private),40.71616168,-73.88307006,-240,Tue Apr 03 18:02:24 +0000 2012


## Table Visit

In [29]:
db.sql(f"""
DROP TABLE IF EXISTS Visit;
CREATE TABLE Visit AS
  SELECT *
  FROM '{local_dir}/dataset_TSMC2014_NYC.csv';
""")
print("done")

done


In [30]:
visit = db.table('Visit')
display(visit, 5)

userId                   venueId           venueCategoryId  \
0     470  49bbd6c0f964a520f4531fe3  4bf58dd8d48988d127951735   
1     979  4a43c0aef964a520c6a61fe3  4bf58dd8d48988d1df941735   
2      69  4c5cc7b485a1e21e00d35711  4bf58dd8d48988d103941735   
3     395  4bc7086715a7ef3bef9878da  4bf58dd8d48988d104941735   
4      87  4cf2c5321d18a143951b5cec  4bf58dd8d48988d1cb941735   

         venueCategory   latitude  longitude  timezoneOffset  \
0  Arts & Crafts Store  40.719810 -74.002581            -240   
1               Bridge  40.606800 -74.044170            -240   
2       Home (private)  40.716162 -73.883070            -240   
3       Medical Center  40.745164 -73.982519            -240   
4           Food Truck  40.740104 -73.989658            -240   

                     utcTimestamp  
0  Tue Apr 03 18:00:09 +0000 2012  
1  Tue Apr 03 18:00:25 +0000 2012  
2  Tue Apr 03 18:02:24 +0000 2012  
3  Tue Apr 03 18:02:41 +0000 2012  
4  Tue Apr 03 18:03:00 +0000 2012

In [31]:
display_schema('Visit')

column_name data_type
0           userId    BIGINT
1          venueId   VARCHAR
2  venueCategoryId   VARCHAR
3    venueCategory   VARCHAR
4         latitude    DOUBLE
5        longitude    DOUBLE
6   timezoneOffset    BIGINT
7     utcTimestamp   VARCHAR

## Nettoyage

Certains utilisateurs ont plusieurs visites au **même instant** (du même POI ou de POI différents), ce qui n'est pas cohérent.

In [32]:
%%sql
SELECT userId, utcTimestamp, count(*) AS nb_visites, count(DISTINCT venueId) AS nb_distinct_poi
FROM Visit
GROUP BY userId, utcTimestamp
HAVING nb_visites > 1
ORDER BY nb_visites DESC
LIMIT 10

userId                    utcTimestamp  nb_visites  nb_distinct_poi
0     739  Thu Apr 19 12:00:12 +0000 2012           4                4
1     739  Fri May 04 12:00:10 +0000 2012           4                4
2     739  Sat Apr 28 12:00:10 +0000 2012           4                4
3     739  Thu May 03 12:00:10 +0000 2012           4                4
4     739  Mon May 14 12:00:11 +0000 2012           4                4
5     344  Mon Nov 19 16:38:14 +0000 2012           4                1
6     739  Tue May 15 12:00:07 +0000 2012           4                4
7     739  Sat Apr 21 12:00:10 +0000 2012           4                4
8     315  Tue Jun 26 23:42:59 +0000 2012           4                1
9     739  Thu Apr 12 12:00:15 +0000 2012           3                3

Plusieurs visites du même lieu pour un utilisateur donné à un instant donné :

In [33]:
%%sql
SELECT *
FROM Visit
WHERE userId = 344
  AND utcTimestamp = 'Mon Nov 19 16:38:14 +0000 2012'

userId                   venueId           venueCategoryId  \
0     344  4d51d1dadcce224bb4f9e51b  4bf58dd8d48988d19b941735   
1     344  4d51d1dadcce224bb4f9e51b  4bf58dd8d48988d19b941735   
2     344  4d51d1dadcce224bb4f9e51b  4bf58dd8d48988d19b941735   
3     344  4d51d1dadcce224bb4f9e51b  4bf58dd8d48988d19b941735   

               venueCategory   latitude  longitude  timezoneOffset  \
0  College Academic Building  40.734901  -73.81892            -300   
1  College Academic Building  40.734901  -73.81892            -300   
2  College Academic Building  40.734901  -73.81892            -300   
3  College Academic Building  40.734901  -73.81892            -300   

                     utcTimestamp  
0  Mon Nov 19 16:38:14 +0000 2012  
1  Mon Nov 19 16:38:14 +0000 2012  
2  Mon Nov 19 16:38:14 +0000 2012  
3  Mon Nov 19 16:38:14 +0000 2012

Plusieurs visites de lieux différents pour un utilisateur donné à un instant donné :

In [34]:
%%sql
SELECT *
FROM Visit
WHERE userId = 739
  AND utcTimestamp = 'Thu Apr 19 12:00:12 +0000 2012'

userId                   venueId           venueCategoryId   venueCategory  \
0     739  4eab92207beb32cb143f9d8d  4bf58dd8d48988d103941735  Home (private)   
1     739  4ba8c654f964a520dbed39e3  4bf58dd8d48988d1fd931735          Subway   
2     739  4f40351fa17cf53abe49a0f4  4bf58dd8d48988d103941735  Home (private)   
3     739  4e8aaccd4fc68f2d71de9384  4bf58dd8d48988d1c7941735     Snack Place   

    latitude  longitude  timezoneOffset                    utcTimestamp  
0  40.818271 -73.906588            -240  Thu Apr 19 12:00:12 +0000 2012  
1  40.816486 -73.907807            -240  Thu Apr 19 12:00:12 +0000 2012  
2  40.818271 -73.906588            -240  Thu Apr 19 12:00:12 +0000 2012  
3  40.815864 -73.907760            -240  Thu Apr 19 12:00:12 +0000 2012

### Question 0 : dédoublonnage

Ne conserver qu'**une seule** visite par couple `(userId, utcTimestamp)` : celle dont le `venueId` est le plus petit.
Le résultat a le même schéma que la table `Visit`.

*Indication* : numéroter les visites à l'intérieur de chaque groupe `(userId, utcTimestamp)` avec `row_number()`.

In [35]:
visit_no_duplicate = db.sql("""
WITH RankedVisits AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY userId, utcTimestamp
            ORDER BY venueId ASC
        ) AS rn
    FROM Visit
)
SELECT * EXCLUDE (rn)
FROM RankedVisits
WHERE rn = 1
""")

display(visit_no_duplicate, 2)

userId                   venueId           venueCategoryId   venueCategory  \
0       1  4a7a7a8ef964a5205de91fe3  4bf58dd8d48988d1c5941735  Sandwich Place   
1       1  4ace6c89f964a52078d020e3  4bf58dd8d48988d1ed931735         Airport   

    latitude  longitude  timezoneOffset                    utcTimestamp  
0  40.719690 -74.008543            -240  Fri May 11 16:10:55 +0000 2012  
1  40.773839 -73.871220            -300  Sat Dec 15 00:13:02 +0000 2012

Vérifier qu'il n'y a plus de doublons (le résultat doit être vide) :

In [36]:
display(db.sql("""
SELECT userId, utcTimestamp, count(*) AS nb
FROM visit_no_duplicate
GROUP BY userId, utcTimestamp
HAVING nb > 1
"""))

Empty DataFrame
Columns: [userId, utcTimestamp, nb]
Index: []

### Remplacer la table Visit par celle sans doublons

In [37]:
nb_visites_avant = db.sql("SELECT count(*) FROM Visit").fetchone()[0]
print("nb de visites avant d'enlever les doublons :", nb_visites_avant)

db.sql("CREATE OR REPLACE TABLE Visit AS SELECT * FROM visit_no_duplicate")

nb_visites_apres = db.sql("SELECT count(*) FROM Visit").fetchone()[0]
print("nb de visites après avoir enlevé les doublons :", nb_visites_apres)

nb de visites avant d'enlever les doublons : 227428
nb de visites après avoir enlevé les doublons : 227102


## Préparation des requêtes

Convertir le timestamp (chaîne de caractères) en date.

In [38]:
visit2 = db.sql("""
  SELECT *, strptime(utcTimestamp, '%a %b %d %X %z %Y') AS datetime
  FROM Visit
""")
display(visit2)

userId                   venueId           venueCategoryId  \
0      950  4c06d0f82e80a5934bb374f9  4bf58dd8d48988d10f951735   
1      950  4c652e64dddfa593196493ff  4d954b06a243a5684965b473   
2      950  4dfd3d04c65b31579b315601  4bf58dd8d48988d163941735   
3      950  4b6dcfd4f964a5202e922ce3  4bf58dd8d48988d175941735   
4      950  4ee781bb4690f8894358d3b6  4d954b0ea243a5684a65b473   
5      951  4f4faefbe4b05c40d6e2174b  4bf58dd8d48988d124941735   
6      951  4f980aa9e4b0d1561acbfc7e  4bf58dd8d48988d112941735   
7      951  4b0737c3f964a520e4f922e3  4bf58dd8d48988d1fd931735   
8      951  4b0737c3f964a520e4f922e3  4bf58dd8d48988d1fd931735   
9      951  4c069dd28a81c9b685ff2590  4bf58dd8d48988d10a951735   
10     951  4b50870af964a5202e2627e3  4bf58dd8d48988d146941735   
11     951  4c799c17bd346dcb2423f6ef  4bf58dd8d48988d103941735   
12     951  4d5ad09923e0a093fb5d5349  4bf58dd8d48988d145941735   
13     951  4f4faefbe4b05c40d6e2174b  4bf58dd8d48988d124941735   
14     951  4aec28cdf964a5206bc521e3  4bf58dd8d48988d1fd931735   
15     951  4b524706f964a5205c7427e3  4bf58dd8d48988d132941735   
16     951  4b538c3cf964a520a4a127e3  4bf58dd8d48988d176941735   
17     951  4b06d110f964a520c3f022e3  4bf58dd8d48988d1fd931735   
18     951  4b0737c3f964a520e4f922e3  4bf58dd8d48988d1fd931735   
19     951  4ae6258af964a52013a521e3  4bf58dd8d48988d1fd931735   
20     951  4d8ca549ca75b60cc859d6a8  4bf58dd8d48988d146941735   
21     951  4c799c17bd346dcb2423f6ef  4bf58dd8d48988d103941735   
22     951  4fac44bae4b03525f2c3b3e0  4bf58dd8d48988d1d0941735   
23     951  4bc9bdf3fb84c9b63a7f1b3e  4bf58dd8d48988d1fc941735   
24     951  4a70a76cf964a52017d81fe3  4bf58dd8d48988d1ab941735   
25     951  4b06d110f964a520c3f022e3  4bf58dd8d48988d1fd931735   
26     951  43bba61df964a520eb2c1fe3  4bf58dd8d48988d118951735   
27     951  50bf9141e4b07113ce781e54  4bf58dd8d48988d13b941735   
28     951  4c799c17bd346dcb2423f6ef  4bf58dd8d48988d103941735   
29     951  4b0737c3f964a520e4f922e3  4bf58dd8d48988d1fd931735   

                               venueCategory   latitude  longitude  \
0                       Drugstore / Pharmacy  40.791402 -74.057030   
1   Residential Building (Apartment / Condo)  40.799581 -73.996224   
2                                       Park  40.805886 -74.044062   
3                       Gym / Fitness Center  40.799776 -74.047980   
4                          Convenience Store  40.790287 -74.058225   
5                                     Office  40.753232 -73.986914   
6                                        Bar  40.753843 -73.987419   
7                                     Subway  40.756506 -73.988028   
8                                     Subway  40.756506 -73.988028   
9                                       Bank  40.753507 -73.987115   
10                             Deli / Bodega  40.754539 -73.988371   
11                            Home (private)  40.861079 -73.930104   
12                        Chinese Restaurant  40.753416 -73.985967   
13                                    Office  40.753356 -73.987333   
14                                    Subway  40.707791 -74.013155   
15                                    Church  40.748769 -73.990624   
16                      Gym / Fitness Center  40.864540 -73.926295   
17                                    Subway  40.858954 -73.932615   
18                                    Subway  40.756506 -73.988028   
19                                    Subway  40.815541 -73.958423   
20                             Deli / Bodega  40.855349 -73.929245   
21                            Home (private)  40.861079 -73.930104   
22                              Dessert Shop  40.742824 -74.000274   
23                           Laundry Service  40.856374 -73.932747   
24                            Student Center  40.738166 -74.001140   
25                                    Subway  40.858801 -73.932586   
26                         Food & Drink Shop  40.734852 -73.991085 

## Exercice 1 : séquences de visites

### 1.0) Numéro de POI

Les identifiants `venueId` sont longs et peu lisibles. Définir :
* `venueId_poi(venueId, poi)` qui associe à chaque `venueId` un numéro `poi` allant de 1 à *n* ;
  *indication* : utiliser `row_number() OVER (…)` ;
* `visit_poi(userId, poi, datetime)`, les visites avec les numéros de POI.

In [ ]:
venueId_poi = db.sql("""
-- Écrire la requête ici
""")
display(db.sql("SELECT * FROM venueId_poi ORDER BY poi"))

In [ ]:
db.sql("SELECT count(*) AS nb_poi FROM venueId_poi")

In [ ]:
visit_poi = db.sql("""
-- Écrire la requête ici
""")
display(db.sql("SELECT * FROM visit_poi ORDER BY datetime"))

### 1.1) Rang

Pour chaque utilisateur, ordonner les visites par date et leur attribuer un **rang** allant de 1 (la plus ancienne) à *n* (la plus récente).

Schéma attendu : `visit_rank(userId, poi, datetime, rank)`

In [ ]:
visit_rank = db.sql("""
-- Écrire la requête ici
""")
display(visit_rank)

### 1.2) Date précédente et durée de déplacement

On veut connaître la durée séparant deux visites consécutives d'un même utilisateur.
Pour chaque visite, ajouter l'attribut **`prev_date`** contenant la date de la visite précédente du même utilisateur.
Pour la première visite d'un utilisateur, `prev_date` est égale à la date de la visite.

Puis ajouter l'attribut **`interval_duration`**, différence en secondes entre la date courante et `prev_date`.

*Indication* : voir la fonction `date_diff('second', a, b)`.

Schémas attendus : `visit_prev_date(userId, poi, datetime, rank, prev_date)`
puis `visit_duration(userId, poi, datetime, rank, prev_date, interval_duration)`

In [ ]:
visit_prev_date = db.sql("""
-- Écrire la requête ici
""")
display(visit_prev_date)

In [ ]:
visit_duration = db.sql("""
-- Écrire la requête ici
""")
display(visit_duration)

### 1.3) Numéro de trajectoire

Découper les visites de chaque utilisateur en trajectoires : dans une trajectoire, la durée entre deux visites
consécutives ne dépasse pas **20 heures** (20 × 3600 secondes).

Ajouter l'attribut **`debut`** valant 1 si la visite commence une trajectoire, 0 sinon,
puis l'attribut **`traj`** indiquant le numéro de la trajectoire (1, 2, …) pour un utilisateur donné.

*Indication* : utiliser une expression `CASE WHEN … THEN … ELSE … END`.

Schémas attendus : `visit_traj1(…, debut)` puis `visit_traj2(…, debut, traj)`

In [ ]:
visit_traj1 = db.sql("""
-- Écrire la requête ici
""")
display(visit_traj1)

In [ ]:
visit_traj2 = db.sql("""
-- Écrire la requête ici
""")
display(visit_traj2)

## Exercice 2 : fenêtres glissantes

### 2.1) Nombre de POI visités dans la semaine précédente

Ajouter l'attribut **`nb_POI_7jours`** donnant le nombre de POI **distincts** visités par l'utilisateur pendant
les 7 jours qui précèdent le jour de la visite courante. Les visites du jour courant ne sont **pas** comptées.

Exemple : pour une visite du 10 avril, on compte les POI visités du 3 au 9 avril inclus.

*Indication* : la fenêtre porte sur des **jours** (`datetime::date`) ; sa borne supérieure précède d'un jour la date courante.

Schéma attendu : `visit_poi_7(userId, poi, datetime, nb_POI_7jours)`

In [ ]:
visit_poi_7 = db.sql("""
-- Écrire la requête ici
""")
display(visit_poi_7)

### 2.2) Nombre cumulé de visites et de POI par utilisateur

Pour chaque visite, ajouter :
* **`cumul_visites`** : le nombre de visites effectuées par l'utilisateur **avant** la visite courante (0 pour la première) ;
* **`cumul_poi`** : le nombre de POI **distincts** déjà visités par l'utilisateur avant la visite courante.

Schéma attendu : `visit_cumul(userId, poi, datetime, rank, cumul_visites, cumul_poi)`

In [ ]:
visit_cumul = db.sql("""
-- Écrire la requête ici
""")
display(visit_cumul)

## Exercice 3 : visites dans une trajectoire

### 3.1) Fusionner les visites consécutives d'un même POI

Une trajectoire peut contenir plusieurs visites **consécutives** d'un même POI : on veut les fusionner en une seule visite.
Pour chaque trajectoire, regrouper les visites consécutives d'un même POI et calculer pour chaque visite fusionnée
sa date de début `date_debut`, sa date de fin `date_fin`, le nombre de check-ins `nb_checkins` et la durée `visit_duration` (en secondes).

* Rmq 1 : si un POI n'a pas de visites consécutives, ses dates de début et de fin sont identiques.
* Rmq 2 : une trajectoire peut contenir plusieurs visites **non consécutives** d'un même POI : elles ne doivent pas être fusionnées.

Schéma attendu : `trajectoire(userId, traj, visitId, poi, nb_checkins, date_debut, date_fin, visit_duration)`,
où `visitId` numérote les visites fusionnées dans la trajectoire.

*Démarche suggérée* : (1) récupérer le POI de la visite précédente dans la trajectoire ;
(2) marquer les visites qui commencent une nouvelle visite fusionnée ; (3) numéroter par somme cumulée ; (4) agréger.

In [ ]:
trajectoire = db.sql("""
-- Écrire la requête ici (on peut la décomposer en plusieurs étapes)
""")
display(trajectoire)

Afficher les visites fusionnées de l'utilisateur 984 :

In [ ]:
display(db.sql("""
SELECT *
FROM trajectoire
WHERE userId = 984
ORDER BY traj, visitId
"""))

### 3.2) Trajectoires contenant une fusion

Afficher toutes les visites (fusionnées) des trajectoires qui contiennent **au moins une** visite fusionnée
(c'est-à-dire avec `nb_checkins > 1`). Trier le résultat par `userId`, `traj`, `date_debut`.

*Indication* : une fenêtre `PARTITION BY userId, traj` permet de calculer une information sur toute la trajectoire
sans perdre le détail des visites.

In [ ]:
%%sql
-- Écrire la requête ici

### 3.3) Statistiques sur les trajectoires

a) Pour chaque trajectoire, calculer le nombre de visites (fusionnées) `nb_visite` et le nombre de POI distincts `nb_poi`.
Stocker le résultat dans une table `stat_traj(userId, traj, nb_visite, nb_poi)`.

b) Calculer le nombre moyen de visites et de POI par trajectoire.

c) Pour chaque valeur de `nb_visite` inférieure à 20, donner le nombre de trajectoires `nb_traj`
et le nombre moyen de POI `avg_nb_poi`, triés par `nb_visite` décroissant.

In [ ]:
# a)
db.sql("""
-- Écrire la requête ici
""")

In [ ]:
%%sql
-- b) Écrire la requête ici

In [ ]:
%%sql
-- c) Écrire la requête ici